# 12.02 · Resilience and cost: retries, fallbacks, rate limits, caching

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lesson 12.01 (SLOs and budgets), Module 03 (middleware) and Lesson 01.03 (`mastery.evals.stats`). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 prompt-cache probe) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Classify** provider errors as transient, permanent or bugs, and retry only the transient ones.
2. **Add** retries with backoff (`ModelRetryMiddleware`) and a fallback model (`ModelFallbackMiddleware`), and measure the success rate they buy, with intervals.
3. **Throttle** calls on the client side with `InMemoryRateLimiter` so the service stays under the provider's quota.
4. **Stop** runaway tool loops with `ModelCallLimitMiddleware`.
5. **Cache** where it is safe (exact model calls), and explain why caching agent answers is risky.
6. **Cut** token cost with provider prompt caching (stable prefix first) and batch jobs, and report cost per model.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) | Model retry (`retry_on`, backoff, `on_failure`), model fallback, model call limit |
| [Models: rate limiting and caching](https://docs.langchain.com/oss/python/langchain/models) | `InMemoryRateLimiter`, model exceptions, response caching |
| [Groq errors](https://console.groq.com/docs/errors) | Which status codes are worth retrying (429, 5xx) and which are not (400, 401, 404) |
| [Groq prompt caching](https://console.groq.com/docs/prompt-caching) | Supported models (gpt-oss), exact-prefix matching, 50% off cached tokens, 2 h expiry |
| [Groq batch](https://console.groq.com/docs/batch) | JSONL batch jobs at 50% off for work that can wait |
| [Groq flex processing](https://console.groq.com/docs/flex-processing) | Flex tier (paid plans only; fails fast with 498 when busy) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-resilience-cost'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · Three kinds of failure

Before adding retries, decide **what** to retry. Provider errors fall into three groups:

| Kind | Examples | Retry? | What to do instead |
|---|---|---|---|
| **Transient** | 429 rate limit, timeout, 500/502/503 | Yes, with backoff | Fallback after the last retry |
| **Permanent** | 401 bad key, 404 unknown model, 400 bad request | No: the next call fails the same way | Fail fast, alert, fix config |
| **Bug** | `KeyError` in your tool or parser | No: retrying hides it and multiplies cost | Let it raise; the trace shows where |

LangChain maps provider errors to standard exceptions in `langchain_core.exceptions`, and each one says whether it is retryable. `mastery.models.is_transient` (Module 02) reads that, and returns `False` for anything it doesn't recognise.

In [ ]:
# 🧪 Offline: what does each kind look like, and is it transient?
from langchain_core.exceptions import ModelAuthenticationError, ModelRateLimitError, ModelTimeoutError

from mastery.service import transient_only

for error in (ModelRateLimitError("429"), ModelTimeoutError("timeout"), ModelAuthenticationError("401"), KeyError("customer_id")):
    print(f"{type(error).__name__:<26} retryable flag={getattr(error, 'is_retryable', '-')!s:<5}  transient_only → {transient_only(error)}")

**You should now see** `True` for the rate limit and the timeout, `False` for the auth error and for the `KeyError`.

### What just happened?

`transient_only` is the predicate we'll give to the retry middleware. Its job is to say **no** to everything that isn't clearly worth another attempt.

---
## Part 2 · Retries with backoff, measured

`flaky(reply, fail_rate=0.3)` is a scripted model response that raises a rate-limit error 30% of the time, with its own seed so runs repeat. We run an agent 60 times without and with `ModelRetryMiddleware` (3 retries, no delay so the notebook is fast; in production keep the exponential backoff and jitter).

In [ ]:
# 🧪 Offline: success rate with and without retries, with 95% intervals.
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

from mastery.evals.stats import wilson_interval
from mastery.service import flaky
from mastery.testing import ScriptedChatModel

def trial(middleware, n=60, fail_rate=0.3):
    good, attempts = 0, 0
    for seed in range(n):
        model = ScriptedChatModel(responses=[flaky("Done.", fail_rate=fail_rate, seed=seed)], repeat_last=True)
        agent = create_agent(model, tools=[], middleware=middleware())
        try:
            agent.invoke({"messages": [{"role": "user", "content": "Where is my invoice?"}]})
            good += 1
        except ModelRateLimitError:
            pass
        attempts += len(model.calls)                      # every attempt is a model call you pay for
    ci = wilson_interval(good, n)
    return {"success": f"{good}/{n}", "95% interval": f"[{ci.low:.2f}, {ci.high:.2f}]", "calls/run": round(attempts / n, 2)}

no_retry = lambda: []
with_retry = lambda: [ModelRetryMiddleware(max_retries=3, retry_on=transient_only, on_failure="error",
                                           initial_delay=0, jitter=False)]   # production: keep the defaults (1 s, ×2, jitter)
table([{"setup": "no retry", **trial(no_retry)}, {"setup": "3 retries", **trial(with_retry)}],
      ["setup", "success", "95% interval", "calls/run"])

**You should now see** `41/60` without retries and `60/60` with them, and about 1.4 model calls per run with retries.

### What just happened?

- With a 30% failure rate, four attempts all fail with probability 0.3⁴ ≈ 0.8%: retries turn a broken service into a reliable one for **transient** errors.
- Retries cost calls (≈1.4 per run here). Under a real rate limit, every retry also counts against the quota, so add backoff and a cap, and watch calls per run.
- `on_failure="error"` re-raises after the last attempt. The documented default is `"continue"`, which ends the run with an AI message describing the error; next you'll see why a service usually wants `"error"`.

In [ ]:
# 🧪 Offline: what does the DEFAULT retry_on do with a bug?
def bug(messages):
    raise KeyError("customer_id")                         # a programming error, not a provider problem

for label, kwargs in (("default retry_on", {}), ("transient_only", {"retry_on": transient_only})):
    model = ScriptedChatModel(responses=[bug], repeat_last=True)
    agent = create_agent(model, tools=[], middleware=[ModelRetryMiddleware(max_retries=3, initial_delay=0, jitter=False, **kwargs)])
    try:
        out = agent.invoke({"messages": [{"role": "user", "content": "hi"}]})
        result = "returned: " + out["messages"][-1].content
    except KeyError:
        result = "raised KeyError"
    print(f"{label:<17} attempts={len(model.calls)}  {result}")

**You should now see** `default retry_on attempts=4  returned: Model call failed after 4 attempts with KeyError: ...` and `transient_only attempts=1  raised KeyError`.

### What just happened?

The documented default (`langchain>=1.3.16`) retries retryable model errors **and every unclassified exception**, and skips model errors marked non-retryable (so a 401 is not retried). A bug is unclassified, so it was retried 4 times, and with `on_failure="continue"` its error text became the "answer" a user would see. In a service: pass an explicit `retry_on`, use `on_failure="error"`, and turn the exception into a clean HTTP error at the edge (Lesson 12.04).

---
## Part 3 · A fallback model

When the primary provider is down, retries only add delay. `ModelFallbackMiddleware(backup, ...)` tries the next model when the primary call fails (after its own retries). The backup can be another Groq model (different capacity pool), another provider (your OpenAI or Gemini key), or a smaller model that gives a shorter answer.

In [ ]:
# 🧪 Offline: the primary always fails; the backup answers.
from langchain.agents.middleware import ModelFallbackMiddleware

primary = ScriptedChatModel(model_name="primary", responses=[flaky("primary", fail_rate=1.0)], repeat_last=True)
backup = ScriptedChatModel(model_name="backup", responses=["Backup answered."], repeat_last=True)
agent = create_agent(primary, tools=[], middleware=[
    ModelFallbackMiddleware(backup),
    ModelRetryMiddleware(max_retries=2, retry_on=transient_only, on_failure="error", initial_delay=0, jitter=False),
])
out = agent.invoke({"messages": [{"role": "user", "content": "Where is my invoice?"}]})
print("answer:", out["messages"][-1].content)
print("calls: primary", len(primary.calls), "· backup", len(backup.calls))

**You should now see** `answer: Backup answered.` and `calls: primary 3 · backup 1`: one attempt plus two retries on the primary, then the backup.

### What just happened?

Middleware order matters: the first in the list wraps the others. Here the fallback wraps the retry, so the primary gets its retries first and the backup is the last resort. Count how often the fallback fires (Lesson 12.07): a fallback that answers 30% of traffic is an outage nobody noticed, and the backup may have different quality, so run your eval set on it too (Module 10).

---
## Part 4 · Stay under the quota: client-side rate limiting

Retries react to 429s; a **rate limiter** avoids them. `InMemoryRateLimiter` is a token bucket: `requests_per_second` refills it, `max_bucket_size` caps bursts. Pass it as `rate_limiter=` to any chat model (`get_model(..., rate_limiter=...)` in this course). Groq's free plan allows `gpt-oss-20b` 30 requests and 8K tokens per minute (check today's numbers in the research checkpoint of 12.01), so 0.5 requests per second, with a cap on `max_tokens`, is a safe setting.

In [ ]:
# 🧪 Offline: 10 calls through a 5-requests-per-second limiter.
import time

from langchain_core.rate_limiters import InMemoryRateLimiter

limiter = InMemoryRateLimiter(requests_per_second=5, check_every_n_seconds=0.01, max_bucket_size=1)
limited = ScriptedChatModel(responses=["ok"], repeat_last=True, rate_limiter=limiter)
start = time.perf_counter()
for _ in range(10):
    limited.invoke("hi")
print(f"10 calls in {time.perf_counter() - start:.1f} s (expected ≈ 2 s at 5 per second)")

**You should now see** `10 calls in 2.0 s` (give or take a tenth).

### What just happened?

The limiter **waits** instead of failing, which moves the cost from errors to latency: that shows up as queue time in the latency budget of 12.01. Two limits to remember: it counts **requests**, not tokens (Groq also limits tokens per minute), and it lives in **one process**. With several workers or containers, give each a share of the quota or use a shared limiter (for example Redis-backed).

---
## Part 5 · Stop runaway loops

A model that keeps calling the same tool never fails: it just costs money and time until the recursion limit. `ModelCallLimitMiddleware` caps model calls per run (`run_limit`) or per thread (`thread_limit`); `exit_behavior="end"` ends the run with a message instead of raising.

In [ ]:
# 🧪 Offline: a model stuck calling `lookup` forever, with and without a call limit.
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.errors import GraphRecursionError

@tool
def lookup(query: str) -> str:
    """Look something up in the help center."""
    return "nothing found"

def stuck():
    return ScriptedChatModel(responses=[AIMessage("", tool_calls=[{"name": "lookup", "args": {"query": "refund"}, "id": "c1"}])],
                             repeat_last=True)

loop = stuck()
try:
    create_agent(loop, tools=[lookup]).invoke({"messages": [{"role": "user", "content": "refund?"}]}, {"recursion_limit": 25})
except GraphRecursionError:
    print(f"no limit   : GraphRecursionError after {len(loop.calls)} model calls")

loop = stuck()
agent = create_agent(loop, tools=[lookup], middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")])
out = agent.invoke({"messages": [{"role": "user", "content": "refund?"}]})
print(f"run_limit=4: {len(loop.calls)} model calls, last message: {out['messages'][-1].content!r}")

**You should now see** `GraphRecursionError after 13 model calls`, then `run_limit=4: 4 model calls` ending with `'Model call limits exceeded: run limit (4/4)'`.

### What just happened?

The limit turns an expensive crash into a cheap, predictable stop. Pick `run_limit` from your traces (the desk needs 2–4 calls for a normal request) plus headroom, and alert when runs hit it: it's a quality signal, not just a cost guard. The message is for logs; show users a friendlier one.

---
## Part 6 · Response caching, and why agents make it risky

LangChain can cache model responses (`cache=InMemoryCache()` per model, or a global cache). The key is the **exact** prompt plus the model's settings. That is safe for deterministic, shared, non-personal calls, such as classifying an FAQ question. It is risky for agents:

| Risk | Example |
|---|---|
| Personal data across users | "What's my balance?" cached for one user, served to another with the same wording |
| Stale tool results | A cached answer that quotes yesterday's invoice status |
| Hidden side effects | A cached tool-calling turn skips the model, but the tool still runs (or doesn't) |
| Sampling | With temperature > 0 the cache freezes one sample forever |

In [ ]:
# 🧪 Offline: two identical prompts hit the cache; a different one doesn't.
from langchain_core.caches import InMemoryCache

cached = ScriptedChatModel(responses=["Answer 1", "Answer 2"], cache=InMemoryCache(maxsize=1000))
print(cached.invoke("How do I reset my password?").content)
print(cached.invoke("How do I reset my password?").content, "(cache hit)")
print(cached.invoke("How do I reset my PASSWORD?").content, "(different text, so a miss)")
print("model calls:", len(cached.calls))

**You should now see** `Answer 1` twice, then `Answer 2`, and `model calls: 2`.

### What just happened?

Exact-match caching only helps when the same text repeats, which is rare for free-form chat. For agents, prefer caching **inside** tools you control (FAQ search results, product catalog lookups) with an explicit expiry and no user data in the key, and leave the model call uncached. The bigger saving is usually the provider's own prefix cache, next.

---
## Part 7 · Provider prompt caching: put the stable part first

Groq caches **prefixes** on `gpt-oss` models automatically: when a new request starts with exactly the same tokens as a recent one (system prompt, tool definitions, examples), that part is billed at 50% and processed faster. Nothing to switch on; you only have to **order** the prompt well: stable content first, user-specific content last. The cache expires after 2 hours without use, and hits are not guaranteed.

In [ ]:
# 🧪 Offline: the same desk prompt built two ways. Which one shares a prefix across users?
from mastery.agents.support import GENERALIST_PROMPT, make_skills

policies = "\n\n".join(skill["content"] for skill in make_skills().values())

def bad_prompt(user, question):
    return f"Customer {user} asks: {question}\n\n{GENERALIST_PROMPT}\n\n{policies}"     # user data first

def good_prompt(user, question):
    return f"{GENERALIST_PROMPT}\n\n{policies}\n\nCustomer {user} asks: {question}"     # stable prefix first

def shared_prefix(a, b):
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    return n

for build in (bad_prompt, good_prompt):
    a, b = build("C-1001", "Why was I charged twice?"), build("C-2042", "How do I export my data?")
    print(f"{build.__name__:<12} shared prefix {shared_prefix(a, b):>5} of {len(a)} characters")

**You should now see** a shared prefix of a handful of characters for `bad_prompt` and over 4,500 for `good_prompt`.

### What just happened?

Same content, different order: one version can reuse about 1,100 tokens of computation per request, the other almost nothing. `create_agent(system_prompt=...)` already puts the system prompt and tool schemas first; what breaks caching is putting timestamps, user names or retrieved documents **into** the system prompt. Keep those in later messages.

### 💸 Probe: read the cached tokens

Two calls with the same long prefix. LangChain reports cached input tokens as `usage_metadata["input_token_details"]["cache_read"]` (from Groq's `prompt_tokens_details.cached_tokens`).

> 💸 Calls `gpt-oss-20b` (a few calls, a few thousand tokens of your free-tier quota). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live = get_model("fast", max_tokens=150)
    usage = []
    for user, question in (("C-1001", "Why was I charged twice?"), ("C-2042", "How do I export my data?")):
        reply = live.invoke(good_prompt(user, question))
        details = reply.usage_metadata.get("input_token_details") or {}
        usage.append(reply.usage_metadata)
        print(f"{user}: input {reply.usage_metadata['input_tokens']:>5}, cached {details.get('cache_read', 0):>5}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** the first call with `cached 0` (or some tokens, if you ran this cell recently) and the second with most of its ~1,100 input tokens cached. A hit is not guaranteed: if both show 0, run the cell again.

### What just happened?

You measured the effect instead of trusting it. In a service, log `cache_read` per run (Lesson 12.07): a hit rate that drops after a deploy usually means someone put a variable into the prefix.

---
## Part 8 · Work that can wait: batch jobs, and what flex is

Not everything needs an answer in seconds. Nightly evals, re-labelling traces or summarising yesterday's tickets can go through Groq's **batch API**: upload a JSONL file, one request per line, and collect results within a window you choose (24 hours to 7 days), at 50% off. **Flex processing** is a different trade (paid plans only): normal latency when there is capacity, an immediate 498 error when there isn't, so it needs the retry logic from Part 2.

We build the batch file offline; submitting it is up to you (check in the console whether your plan includes batch).

In [ ]:
# 🧪 Offline: a batch file for a nightly eval of 20 support questions (format from the Groq batch docs).
import json

from mastery.agents.support_traces import generate_inputs

OUT = REPO_ROOT / "data" / "outputs" / "m12"
OUT.mkdir(parents=True, exist_ok=True)
batch_path = OUT / "nightly-batch.jsonl"
lines = [{"custom_id": f"eval-{i:03d}", "method": "POST", "url": "/v1/chat/completions",
          "body": {"model": settings.model_fast, "max_tokens": 300,
                   "messages": [{"role": "system", "content": GENERALIST_PROMPT}, {"role": "user", "content": item["turns"][0]}]}}
         for i, item in enumerate(generate_inputs(20, seed=12))]
batch_path.write_text("\n".join(json.dumps(line) for line in lines) + "\n", encoding="utf-8")
print(batch_path.relative_to(REPO_ROOT), "·", len(lines), "requests ·", batch_path.stat().st_size, "bytes")
print(lines[0]["custom_id"], lines[0]["body"]["messages"][1]["content"][:70])

**You should now see** `data/outputs/m12/nightly-batch.jsonl · 20 requests` and the first question.

### What just happened?

Each line is a complete chat-completions request with a `custom_id` you use to match results back. Keep the system prompt identical across lines: batch requests also benefit from the prefix cache. Results arrive as another JSONL file; treat it like any eval run (Module 10).

---
## Part 9 · Cost per model, from usage metadata

Every chat model reply carries `usage_metadata`; `get_usage_metadata_callback()` adds it up **per model** across a block of code (for real providers), and `mastery.pricing.cost_report` prices it. Run the Project 09 desk offline on 30 inputs and price the tokens as if they had gone to `gpt-oss-20b`.

In [ ]:
# 🧪 Offline: add up usage_metadata of every model reply in 30 desk runs, priced at gpt-oss-20b list prices.
from langchain_core.messages import AIMessage
from langchain_core.messages.ai import add_usage
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.pricing import cost_report

desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
total = None
for i, item in enumerate(generate_inputs(30, seed=12)):
    out = app.invoke({"messages": [{"role": "user", "content": item["turns"][0]}]}, {"configurable": {"thread_id": f"cost-{i}"}})
    for message in out["messages"]:
        if isinstance(message, AIMessage) and message.usage_metadata:
            total = add_usage(total, message.usage_metadata)

# Real providers set response_metadata["model_name"], so get_usage_metadata_callback() groups usage per model
# for you; the offline model doesn't, so we name the model whose prices we want.
rows = cost_report({settings.model_fast: total})
table(rows, ["model", "input_tokens", "output_tokens", "total_tokens", "cost_usd"])
per_run = rows[0]["cost_usd"] / 30 if rows and rows[0]["cost_usd"] else None
print("cost per run:", f"${per_run:.6f}" if per_run else "unpriced")

**You should now see** one `openai/gpt-oss-20b` row with about 31,500 input and 1,500 output tokens, $0.0028 in total, and `cost per run: $0.000094`. Input dominates: every model call resends the conversation and the tool schemas, which is exactly what the prefix cache discounts. The offline model's token counts are estimates (the scripted model counts characters), so treat the absolute number as an order of magnitude; the method is what carries over.

### What just happened?

This is the per-run cost the ledger in 12.01 needs. In the service (Lesson 12.04) the same callback wraps each run, and the ledger is charged with the real number instead of the estimate. LangSmith computes the same costs per trace from its model price table (see the cost tracking page in 12.01's reading list).

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| A bug shows up as "Model call failed after 4 attempts" in the chat | Default `retry_on` retries unclassified errors; `on_failure="continue"` | Explicit `retry_on=transient_only`, `on_failure="error"` |
| 429s get worse with retries | No backoff, or every worker retries at once | Exponential backoff with jitter; a client-side rate limiter |
| Fallback answers most traffic for days | Nobody counts fallbacks | Metric and alert on fallback rate (12.07) |
| Rate limiter set, still 429 | Limit per process, several workers; or tokens per minute exceeded | Split the quota across workers; cap `max_tokens` |
| User sees another user's answer | Response cache keyed on text only | Don't cache personal model calls; cache inside tools with explicit keys |
| `cache_read` always 0 | Variable content (time, name) at the start of the prompt; non-gpt-oss model | Stable prefix first (Part 7); use a supported model |
| Agent hits `GraphRecursionError` | Tool loop | `ModelCallLimitMiddleware(run_limit=...)`; fix the prompt or tool |

## ⚠️ Production notes

> ⚠️ **Production note: retries multiply load during an outage.**
> Three retries on every request turn a provider brown-out into 4× traffic. Cap total retries per second (a retry budget), back off, and fail over to the fallback rather than retrying forever.

> ⚠️ **Production note: idempotency before retries for side effects.** (Part 2)
> Model retries are safe because a model call has no side effects. A **tool** that issues a refund is not: never retry it blindly. Lesson 12.04 adds idempotency keys so a repeated request can't refund twice.

## 🏋️ Exercises

1. **(Easy)** Re-run Part 2 with `fail_rate=0.6`. How many retries do you need for 59 of 60 runs to succeed? Check against 1 − 0.6ᵏ.
2. **(Medium)** Make the fallback in Part 3 a **real** second model behind a flag: `get_model("reasoning")` or your OpenAI key. What changes in cost per answer?
3. **(Medium)** Write a `retry_on` that also retries `ModelAPIError` with status 503, but not 500. Where would you read the status from?
4. **(Stretch)** Cache FAQ search results inside a tool with a 10-minute expiry (`functools.lru_cache` is not enough: it never expires). Show a hit and an expiry with a fake clock.

<details>
<summary><b>Show solutions</b></summary>

**1.** 1 − 0.6ᵏ ≥ 59/60 needs 0.6ᵏ ≤ 0.017, so k ≥ 8 attempts (7 retries). With 3 retries expect about 52 of 60. High failure rates are a reason to fall back, not to retry more.

**2.**
```python
if HAVE_GROQ_KEY:
    backup = get_model("reasoning", max_tokens=300)
```
`gpt-oss-120b` costs several times more per token, so the fallback rate directly drives your bill: put it in the cost report.

**3.**
```python
from langchain_core.exceptions import ModelAPIError
def retry_on(error):
    if transient_only(error):
        return True
    status = getattr(error, "status_code", None) or getattr(getattr(error, "response", None), "status_code", None)
    return isinstance(error, ModelAPIError) and status == 503
```
The status lives on the provider exception (`status_code`, or the HTTP `response`); check what your integration sets with a quick test.

**4.**
```python
import time
class TTLCache:
    def __init__(self, ttl, clock=time.monotonic):
        self.ttl, self.clock, self.data = ttl, clock, {}
    def get(self, key, compute):
        value, at = self.data.get(key, (None, -1e18))
        if self.clock() - at > self.ttl:
            value = compute()
            self.data[key] = (value, self.clock())
        return value
now = [0.0]
cache = TTLCache(600, clock=lambda: now[0])
cache.get("reset", lambda: "v1"); now[0] = 300; print(cache.get("reset", lambda: "v2"))   # v1 (hit)
now[0] = 700; print(cache.get("reset", lambda: "v3"))                                      # v3 (expired)
```
</details>

## 🔎 Docs drill (5 min)

What is the documented default of `ModelRetryMiddleware`'s `on_failure`, and what does the run return when retries are exhausted with that default?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) and find **Model retry** → the `on_failure` parameter.
2. The default is `"continue"`: the run ends with an `AIMessage` that describes the error instead of raising.
3. Confirm in the installed source: `langchain/agents/middleware/model_retry.py` (search for `on_failure`), and compare with what Part 2 printed.

</details>

## ✅ Checkpoint

- [ ] I can say which errors are transient, permanent or bugs, and retry only the transient ones.
- [ ] I measured success rate with and without retries, with intervals, and the extra calls retries cost.
- [ ] I added a fallback model and know where middleware order matters.
- [ ] I throttled calls with a rate limiter and know it is per process.
- [ ] I stopped a tool loop with a call limit.
- [ ] I can explain when response caching is safe, and ordered a prompt for the provider's prefix cache.
- [ ] I built a batch file and a cost report.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Groq prompt caching](https://console.groq.com/docs/prompt-caching) and [Groq batch](https://console.groq.com/docs/batch) and log in [`docs/research-log.md`](../../docs/research-log.md): which models support prompt caching (2026-10-16: the gpt-oss family), the discount (50%), and whether batch is available on your plan.

## Summary

- Retry transient errors only, with backoff; `retry_on` and `on_failure` must be explicit in a service.
- A fallback model covers outages; count how often it fires.
- A rate limiter trades errors for latency, per process; a call limit caps runaway loops.
- Cache exact model calls only when shared and non-personal; prefer caching inside tools.
- Order prompts stable-first for provider prefix caching; push waiting work to batch; price every run.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-03-async-and-concurrency.ipynb** → [`12-03-async-and-concurrency.ipynb`](12-03-async-and-concurrency.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)